In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import time

# Load data
print("1. Loading data...")
df = pd.read_csv('Nashville_Accidents_Transformed.csv')
df.drop(columns=[col for col in ['ID', 'Severity'] if col in df.columns], inplace=True)

# Prepare target
print("2. Preparing target...")
y = df['Hit and Run'].str.lower().map({'y': 1, 'n': 0})
X = df.drop('Hit and Run', axis=1)

# Split data
print("3. Splitting data...")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Preprocessing
print("4. Setting up preprocessor...")
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()

# Modified preprocessor with memory optimization
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features)  # Keep sparse
    ]
)

# Fit transform with timing
print("5. Fitting preprocessor...")
start_time = time.time()
X_train = preprocessor.fit_transform(X_train_raw)
print(f"Preprocessing completed in {time.time()-start_time:.2f} seconds")

# Check matrix size
print(f"X_train shape: {X_train.shape} (sparse: {hasattr(X_train, 'toarray')})")

# SMOTE with error handling
print("6. Applying SMOTE...")
try:
    smote = SMOTE(random_state=42)
    start_time = time.time()
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    print(f"SMOTE completed in {time.time()-start_time:.2f} seconds")
    print(f"Resampled shape: {X_resampled.shape}")
except Exception as e:
    print(f"SMOTE failed: {str(e)}")
    print("Continuing without SMOTE...")
    X_resampled, y_resampled = X_train, y_train

# Model training
print("7. Training model...")
model = LogisticRegression(max_iter=1000, solver='liblinear', verbose=1)  # Added verbose
start_time = time.time()
model.fit(X_resampled, y_resampled)
print(f"Model trained in {time.time()-start_time:.2f} seconds")

# Evaluation
print("8. Evaluating...")
X_test = preprocessor.transform(X_test_raw)
y_pred = model.predict(X_test)

print("\n=== Results ===")
print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

1. Loading data...
2. Preparing target...
3. Splitting data...
4. Setting up preprocessor...
5. Fitting preprocessor...
Preprocessing completed in 5.12 seconds
X_train shape: (151003, 171324) (sparse: True)
6. Applying SMOTE...
SMOTE completed in 271.88 seconds
Resampled shape: (222998, 171324)
7. Training model...
[LibLinear]Model trained in 5.07 seconds
8. Evaluating...

=== Results ===
[[47786     0]
 [    0 16930]]
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     47786
           1       1.00      1.00      1.00     16930

    accuracy                           1.00     64716
   macro avg       1.00      1.00      1.00     64716
weighted avg       1.00      1.00      1.00     64716

